# Cost-Effectiveness Scoring : Energy Storage Units

This notebook builds a cost-effectiveness scoring model for 25 battery 
storage units connected across a 5-bus power grid.

Each storage unit is scored and ranked based on three characteristics:

- Efficiency ; how much energy is retained per charge/discharge cycle
- Energy Capacity ; how much energy the unit can store
- Duration ; how long the unit can discharge at full power

The combined score allows us to identify which storage units deliver 
the best overall value ; and which are underperforming relative to 
their capacity.

**Scoring formula:**
Score = (Efficiency × 0.4) + (Normalised Capacity × 0.4) + (Normalised Duration × 0.2)

## Setup : Load and Prepare Data

---

In [2]:
import pandas as pd
import plotly.express as px

# Load data
df1 = pd.read_excel("../data/stordata_capacity_vlookup.xlsx",
                    sheet_name="Sheet1",
                    usecols=[0,1,2,3])

df2 = pd.read_excel("../data/stordata_capacity_vlookup.xlsx",
                    sheet_name="Sheet2",
                    usecols=[1,2])

df2.set_index("id", inplace=True)
df1["power capability"] = df1["id"].map(df2["power capability"])
df1["energy capacity (kWh)"] = df1["power capability"] * df1["duration"]
df1["bus"] = df1["bus"].str.replace("bus 2", "bus_2")

df1.head()

,id,bus,Efficiency,duration,power capability,energy capacity (kWh)
0,stor1,bus_3,0.76,4.0,8.0,32.0
1,stor2,bus_3,0.87,1.0,5.0,5.0
2,stor3,bus_3,0.85,0.5,18.0,9.0
3,stor4,bus_2,0.89,4.0,12.0,48.0
4,stor5,bus_2,0.87,3.0,5.0,15.0


## Step 1 : Normalise the Data

Before scoring, we normalise energy capacity and duration to a 0 to 1 scale. 
This ensures no single metric dominates the score simply because 
its numbers are larger than the others. This converts both to the same
scale first.

**Normalisation formula:**

Normalised Value = (Value - Minimum) / (Maximum - Minimum)

A value of 0 means the lowest in the dataset.
A value of 1 means the highest in the dataset.

In [3]:
# Normalise energy capacity and duration
df1["normalised capacity"] = (
    (df1["energy capacity (kWh)"] - df1["energy capacity (kWh)"].min()) /
    (df1["energy capacity (kWh)"].max() - df1["energy capacity (kWh)"].min())
)

df1["normalised duration"] = (
    (df1["duration"] - df1["duration"].min()) /
    (df1["duration"].max() - df1["duration"].min())
)

df1[["id", "Efficiency", "energy capacity (kWh)", 
     "normalised capacity", "duration", "normalised duration"]].head(10)

,id,Efficiency,energy capacity (kWh),normalised capacity,duration,normalised duration
0,stor1,0.76,32.0,0.056530,4.0,0.411765
1,stor2,0.87,5.0,0.003899,1.0,0.058824
2,stor3,0.85,9.0,0.011696,0.5,0.000000
3,stor4,0.89,48.0,0.087719,4.0,0.411765
4,stor5,0.87,15.0,0.023392,3.0,0.294118
5,stor6,0.85,76.0,0.142300,2.0,0.176471
6,stor7,0.85,51.0,0.093567,1.5,0.117647
7,stor8,0.90,3.0,0.000000,1.0,0.058824
8,stor9,0.88,11.0,0.015595,0.5,0.000000
9,stor10,0.65,8.0,0.009747,4.0,0.411765


## Step 2 : Calculate the Cost-Effectiveness Score

Each storage unit receives a combined score based on 
three weighted characteristics:

| Characteristic | Weight | Reason |
|---|---|---|
| Efficiency | 40% | Energy waste costs money every cycle |
| Normalised Capacity | 40% | More storage delivers more value |
| Normalised Duration | 20% | Longer duration is beneficial but less critical |


**Important note on weights:**
These weights represent one possible decision framework ; not a universal 
ranking method. Different project objectives could justify different weights. 
For example:

- A grid prioritising long duration backup storage would increase 
  the duration weight significantly
- A grid operator focused purely on minimising energy waste would 
  increase the efficiency weight
- A project with tight capital budget constraints would incorporate 
  cost per kWh as an additional weighted factor

The analysis should therefore be interpreted as a starting point for 
decision making ; not a definitive ranking.

---

In [4]:
# Calculate cost-effectiveness score
df1["score"] = (
    (df1["Efficiency"] * 0.4) +
    (df1["normalised capacity"] * 0.4) +
    (df1["normalised duration"] * 0.2)
)

# Rank storage units by score
df1_ranked = df1[["id", "bus", "Efficiency", 
                   "energy capacity (kWh)", "duration", "score"]].sort_values(
    "score", ascending=False).reset_index(drop=True)

df1_ranked.index += 1  # starts the ranking from 1 instead of 0. So rank 1 is the best storage unit, rank 25 is the worst
df1_ranked

,id,bus,Efficiency,energy capacity (kWh),duration,score
1,stor11,bus_5,0.85,516.0,6.0,0.869412
2,stor24,bus_3,0.75,468.0,9.0,0.862573
3,stor21,bus_1,0.82,376.0,4.0,0.701191
4,stor20,bus_1,0.81,304.0,4.0,0.641051
5,stor23,bus_1,0.85,256.0,4.0,0.619624
6,stor17,bus_4,0.90,115.5,3.5,0.518308
7,stor19,bus_5,0.65,180.0,4.0,0.480365
8,stor4,bus_2,0.89,48.0,4.0,0.473441
9,stor16,bus_2,0.77,100.0,4.0,0.465986
10,stor15,bus_2,0.85,48.0,4.0,0.457441


<span style="color:#c0392b;">**Data Quality Issue:**</span>

- stor999 returns NaN for score because its power capability data 
  was missing in the original dataset.
- Without power capability, energy capacity cannot be computed ; 
  and without energy capacity the score cannot be calculated.
- In a real grid analysis, stor999 would need to be investigated 
  and its missing data resolved before inclusion in the ranking.

<span style="color:orange;">**Analysis : Cost-Effectiveness Ranking**</span>

- stor11 on bus_5 ranks 1st with a score of 0.87 ; driven by 
  the highest energy capacity (516 kWh) and longest duration (6 hours) 
  in the dataset.
- stor24 ranks 2nd despite having low efficiency (0.75) ; its 
  exceptional capacity (468 kWh) compensates for poor efficiency. 
  This highlights a limitation of capacity-weighted scoring ; 
  large but inefficient units can still rank highly.
- stor17 on bus_4 ranks 6th with the highest efficiency in the 
  dataset (0.90) but limited capacity (115.5 kWh) ; showing that 
  efficiency alone is not enough to achieve a top ranking.
- The bottom ranked storage units are mostly short duration ; 
  low capacity units ; confirming that size and duration matter 
  more than efficiency in this scoring model.

---
## Visualisation : Cost-Effectiveness Rankings



In [5]:
# Bar chart : cost-effectiveness ranking
fig = px.bar(
    df1_ranked.dropna(subset=["score"]),
    x="id",
    y="score",
    color="bus",
    hover_data=["Efficiency", "energy capacity (kWh)", "duration"],
    title="Cost-Effectiveness Score by Storage Unit ; Ranked Best to Worst",
    labels={
        "id": "Storage Unit",
        "score": "Cost-Effectiveness Score"
    }
)

fig.update_layout(xaxis_tickangle=45)
fig.show()

<span style="color:grey;">*Figure 1 : Cost-effectiveness scores for 24 storage units ranked 
from best to worst. Bars are coloured by bus. stor999 is excluded 
due to missing power capability data. Hover over any bar to see 
the full profile of that storage unit.*</span>

### Chart 2

In [6]:
# Scatter plot : score vs efficiency
fig = px.scatter(
    df1_ranked.dropna(subset=["score"]),
    x="Efficiency",
    y="score",
    color="bus",
    size="energy capacity (kWh)",
    hover_data=["id", "duration"],
    title="Cost-Effectiveness Score vs Efficiency",
    labels={
        "score": "Cost-Effectiveness Score",
        "Efficiency": "Efficiency"
    }
)

fig.show()

<span style="color:grey;">*Figure 2 : Each dot represents one storage unit plotted by efficiency 
and cost-effectiveness score. Dot size represents energy capacity ; 
larger dots indicate higher capacity storage units. Notice that high 
efficiency does not guarantee a high score ; stor11 and stor24 achieve 
top scores despite modest or low efficiency because their large capacity 
compensates. Hover over any dot to see the full storage unit profile.*</span>

## Conclusions

This notebook built a cost-effectiveness scoring model for 25 battery 
storage units using a weighted combination of efficiency, energy capacity 
and duration.

**Key findings:**

- stor11 on bus_5 ranks 1st with a score of 0.87 ; the highest energy 
  capacity (516 kWh) and longest duration (6 hours) in the dataset 
  drive its top ranking despite a moderate efficiency of 0.85.

- stor24 ranks 2nd despite having low efficiency (0.75) ; its 
  exceptional capacity (468 kWh) compensates. This reveals a limitation 
  of capacity-weighted scoring ; large but inefficient units can still 
  rank highly.

- stor17 has the highest efficiency in the dataset (0.90) but ranks 
  only 6th ; proving that efficiency alone is insufficient for a 
  top ranking without sufficient capacity to support it.

- stor999 could not be scored due to missing power capability data ; 
  highlighting the critical importance of data completeness in any 
  real grid analysis.

- bus_5 and bus_1 dominate the top rankings ; confirming findings 
  from Notebooks 1 and 2 that these buses host the strongest 
  performing storage units overall.

**Engineering implication:**
A cost-effectiveness score is a practical tool for grid operators 
making investment decisions. Storage units with low scores ; 
particularly those combining low efficiency with low capacity ; 
should be prioritised for replacement or operational optimisation. 
However, the weighting of the scoring formula must reflect the 
specific priorities of the grid operator ; a grid prioritising 
long duration backup storage would weight duration more heavily 
than this model does.

**Limitation of this model:**
This scoring model does not include actual cost data. In a real 
analysis, capital cost (£/kWh) and operational cost would be 
included to produce a true cost-effectiveness ratio. This is 
a logical next step for further analysis.

## Next Steps
- Incorporate real capital cost data (£/kWh) into the scoring model
- Build a Streamlit dashboard allowing interactive adjustment 
  of scoring weights
- Apply this scoring methodology to the thesis dataset for 
  North Cyprus community storage analysis